# Generalized Binder iPSAE Notebook

This Colab notebook recreates the useful idea from Adaptyv Bio's Nipah notebook, but makes the target configurable.

You can use it to:
- prepare Boltz YAML inputs for any target protein
- optionally run Boltz in Colab
- score predicted complexes with `ipSAE`
- batch-rank many binder designs against the same target

Reference workflows:
- [adaptyvbio/nipah_ipsae_pipeline](https://github.com/adaptyvbio/nipah_ipsae_pipeline)
- [DunbrackLab/IPSAE](https://github.com/DunbrackLab/IPSAE)
- [Boltz prediction docs](https://github.com/jwohlwend/boltz/blob/main/docs/prediction.md)

GPU runtime is strongly recommended if you want to run Boltz inside Colab.

In [ ]:
#@title Install dependencies
#@markdown This installs a Colab-safe dependency set and downloads the reference `ipsae.py` scorer.

import os
import subprocess
from pathlib import Path
from urllib.request import urlretrieve

WORKSPACE = Path('/content/binder_ipsae_colab')
WORKSPACE.mkdir(parents=True, exist_ok=True)
os.chdir(WORKSPACE)

# pandas 2.2.2 is the first pandas release documented as generally compatible
# with both NumPy 1.x and 2.x wheels. Pinning NumPy 1.26.4 keeps Colab stable
# and avoids the common binary-mismatch import error.
marker = WORKSPACE / '.deps_installed_v2'
if not marker.exists():
    subprocess.run([
        'python', '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall',
        'numpy==1.26.4',
        'pandas==2.2.2',
        'pyyaml',
        'biopython',
        'boltz',
    ], check=True)
    marker.write_text('ok\n')
    print('Installed compatible dependencies. Restarting the runtime once to load the new binaries...')
    os.kill(os.getpid(), 9)

ipsae_url = 'https://raw.githubusercontent.com/adaptyvbio/nipah_ipsae_pipeline/main/ipsae.py'
ipsae_path = WORKSPACE / 'ipsae.py'
urlretrieve(ipsae_url, ipsae_path)

print('Workspace:', WORKSPACE)
print('ipSAE script:', ipsae_path)


In [ ]:
import csv
import json
import re
import shutil
import subprocess
from pathlib import Path

import pandas as pd


AMINO_ACID_3_TO_1 = {
    'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
    'GLN': 'Q', 'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
    'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
    'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V', 'MSE': 'M'
}

SUMMARY_NUMERIC_COLUMNS = [
    'ipSAE', 'ipSAE_d0chn', 'ipSAE_d0dom', 'ipTM_af', 'ipTM_d0chn',
    'pDockQ', 'pDockQ2', 'LIS', 'n0res', 'n0chn', 'n0dom',
    'd0res', 'd0chn', 'd0dom', 'nres1', 'nres2', 'dist1', 'dist2'
]


def normalize_sequence(sequence: str) -> str:
    cleaned = re.sub(r'\s+', '', sequence).upper()
    if not cleaned:
        raise ValueError('Sequence is empty.')
    invalid = sorted({char for char in cleaned if not char.isalpha()})
    if invalid:
        raise ValueError(f'Sequence contains invalid characters: {"".join(invalid)}')
    return cleaned


def sanitize_name(name: str) -> str:
    safe = re.sub(r'[^A-Za-z0-9._-]+', '_', name).strip('_')
    return safe or 'binder'


def read_fasta(path: str | Path):
    path = Path(path)
    records = []
    current_name = None
    current_lines = []
    with path.open() as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if not line:
                continue
            if line.startswith('>'):
                if current_name is not None:
                    records.append({'name': current_name, 'sequence': normalize_sequence(''.join(current_lines))})
                current_name = line[1:].strip() or f'binder_{len(records) + 1}'
                current_lines = []
            else:
                current_lines.append(line)
    if current_name is not None:
        records.append({'name': current_name, 'sequence': normalize_sequence(''.join(current_lines))})
    return records


def extract_sequence_from_pdb(pdb_path: str | Path, chain_id: str = 'A') -> str:
    pdb_path = Path(pdb_path)
    residues = []
    seen = set()
    with pdb_path.open() as handle:
        for line in handle:
            if not line.startswith('ATOM'):
                continue
            if line[21].strip() != chain_id:
                continue
            atom_name = line[12:16].strip()
            if atom_name != 'CA':
                continue
            residue_number = line[22:27].strip()
            if residue_number in seen:
                continue
            seen.add(residue_number)
            residue_name = line[17:20].strip().upper()
            residues.append(AMINO_ACID_3_TO_1.get(residue_name, 'X'))
    return ''.join(residues)


def load_target_sequence(target_mode, target_sequence, target_fasta_path, target_pdb_path, target_pdb_chain):
    if target_mode == 'sequence':
        return normalize_sequence(target_sequence)
    if target_mode == 'fasta':
        records = read_fasta(target_fasta_path)
        if not records:
            raise ValueError(f'No target sequence found in {target_fasta_path}')
        return records[0]['sequence']
    if target_mode == 'pdb':
        sequence = extract_sequence_from_pdb(target_pdb_path, target_pdb_chain)
        if not sequence:
            raise ValueError(f'Could not extract target sequence from {target_pdb_path} chain {target_pdb_chain}')
        return sequence
    raise ValueError(f'Unsupported target_mode: {target_mode}')


def load_binders(binder_sequence, binder_name, binder_fasta_path, binder_csv_path='', binder_name_column='Design', binder_sequence_column='Sequence', binder_rank_column='Rank'):
    binders = []
    if binder_csv_path:
        df = pd.read_csv(binder_csv_path)
        required = [binder_name_column, binder_sequence_column]
        missing = [column for column in required if column not in df.columns]
        if missing:
            raise ValueError(f'Missing binder CSV columns: {missing}')
        if binder_rank_column and binder_rank_column in df.columns:
            df = df.sort_values(binder_rank_column)
        for _, row in df.iterrows():
            name = str(row[binder_name_column]).strip()
            sequence = str(row[binder_sequence_column]).strip()
            if not name or not sequence:
                continue
            rank_prefix = ''
            if binder_rank_column and binder_rank_column in df.columns:
                rank_value = str(row[binder_rank_column]).strip()
                if rank_value:
                    rank_prefix = f'{rank_value}_'
            binders.append({'name': f'{rank_prefix}{name}', 'sequence': normalize_sequence(sequence)})
    if binder_fasta_path:
        binders.extend(read_fasta(binder_fasta_path))
    if binder_sequence.strip():
        binders.append({'name': binder_name, 'sequence': normalize_sequence(binder_sequence)})
    if not binders:
        raise ValueError('Provide a binder CSV path, binder FASTA path, or a single binder sequence.')
    return binders


def yaml_quote(value: str) -> str:
    return "'" + value.replace("'", "''") + "'"


def render_boltz_yaml(target_sequence, binder_sequence, target_chain_id='A', binder_chain_id='B', target_msa_path='', binder_msa_path=''):
    target_msa_value = str(Path(target_msa_path).resolve()) if target_msa_path else 'empty'
    binder_msa_value = str(Path(binder_msa_path).resolve()) if binder_msa_path else 'empty'
    lines = [
        'version: 1',
        'sequences:',
        '  - protein:',
        f'      id: {yaml_quote(target_chain_id)}',
        f'      sequence: {yaml_quote(target_sequence)}',
        f'      msa: {yaml_quote(target_msa_value)}' if target_msa_path else '      msa: empty',
        '  - protein:',
        f'      id: {yaml_quote(binder_chain_id)}',
        f'      sequence: {yaml_quote(binder_sequence)}',
        f'      msa: {yaml_quote(binder_msa_value)}' if binder_msa_path else '      msa: empty',
        ''
    ]
    return '\n'.join(lines)


def prepare_inputs(work_dir, target_sequence, binders, target_name='target', target_chain_id='A', binder_chain_id='B', target_msa_path='', binder_msa_path=''):
    work_dir = Path(work_dir)
    inputs_dir = work_dir / 'inputs'
    inputs_dir.mkdir(parents=True, exist_ok=True)
    manifest_rows = []
    for index, binder in enumerate(binders, start=1):
        job_name = f'{index:03d}_{sanitize_name(binder["name"])}'
        yaml_path = inputs_dir / f'{job_name}.yaml'
        yaml_path.write_text(render_boltz_yaml(target_sequence, binder['sequence'], target_chain_id, binder_chain_id, target_msa_path, binder_msa_path))
        manifest_rows.append({
            'job_name': job_name,
            'target_name': target_name,
            'target_chain_id': target_chain_id,
            'binder_name': binder['name'],
            'binder_chain_id': binder_chain_id,
            'binder_length': len(binder['sequence']),
            'yaml_path': str(yaml_path.resolve())
        })
    manifest_path = work_dir / 'binder_manifest.csv'
    pd.DataFrame(manifest_rows).to_csv(manifest_path, index=False)
    return manifest_path


def run_boltz(input_dir, out_dir, use_msa_server=True, use_potentials=False, override=False, devices=1, recycling_steps=None, diffusion_samples=None):
    boltz_executable = shutil.which('boltz') or 'boltz'
    command = [boltz_executable, 'predict', str(Path(input_dir).resolve()), '--out_dir', str(Path(out_dir).resolve()), '--devices', str(devices)]
    if use_msa_server:
        command.append('--use_msa_server')
    if use_potentials:
        command.append('--use_potentials')
    if override:
        command.append('--override')
    if recycling_steps is not None:
        command.extend(['--recycling_steps', str(recycling_steps)])
    if diffusion_samples is not None:
        command.extend(['--diffusion_samples', str(diffusion_samples)])
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)


def format_cutoff(value):
    text = str(int(value))
    return f'0{text}' if value < 10 else text


def predicted_table_path(structure_path, pae_cutoff, dist_cutoff):
    structure_path = Path(structure_path)
    base = structure_path.with_suffix('')
    return Path(f'{base}_{format_cutoff(pae_cutoff)}_{format_cutoff(dist_cutoff)}.txt')


def summarize_score_table(table_path, job_name):
    df = pd.read_csv(table_path)
    rows = []
    for (_, max_row) in df[df['Type'] == 'max'].iterrows():
        pair = '-'.join(sorted([str(max_row['Chn1']), str(max_row['Chn2'])]))
        asym_mask = ((df['Chn1'] == max_row['Chn1']) & (df['Chn2'] == max_row['Chn2']) & (df['Type'] == 'asym'))
        asym_rows = df[asym_mask]
        for summary_kind in ['max', 'min']:
            row = {'job_name': job_name, 'pair': pair, 'summary_kind': summary_kind}
            for column in SUMMARY_NUMERIC_COLUMNS:
                row[column] = float(max_row[column]) if summary_kind == 'max' else float(asym_rows[column].min())
            rows.append(row)
    return rows


def score_prediction(pae_path, structure_path, ipsae_path, pae_cutoff=15.0, dist_cutoff=15.0, job_name='prediction'):
    command = ['python', str(Path(ipsae_path).resolve()), str(Path(pae_path).resolve()), str(Path(structure_path).resolve()), str(pae_cutoff), str(dist_cutoff)]
    subprocess.run(command, check=True)
    table_path = predicted_table_path(structure_path, pae_cutoff, dist_cutoff)
    return summarize_score_table(table_path, job_name)


def score_batch(predictions_root, ipsae_path, model_index=0, pae_cutoff=15.0, dist_cutoff=15.0):
    predictions_root = Path(predictions_root)
    rows = []
    for structure_path in sorted(predictions_root.glob(f'predictions/*/*_model_{model_index}.cif')):
        pae_path = structure_path.with_name(f'pae_{structure_path.stem}.npz')
        job_name = structure_path.stem.replace(f'_model_{model_index}', '')
        rows.extend(score_prediction(pae_path, structure_path, ipsae_path, pae_cutoff, dist_cutoff, job_name))
    if not rows:
        raise ValueError(f'No prediction CIFs found under {predictions_root}')
    return pd.DataFrame(rows)


In [ ]:
#@title Configure run
#@markdown Upload your files first with the next cell, or point these paths at files already in `/content`.

mode = 'prepare' #@param ['prepare', 'predict_and_score', 'score_existing_predictions']
work_dir = '/content/binder_ipsae_runs' #@param {type:'string'}
target_name = 'target' #@param {type:'string'}

target_mode = 'pdb' #@param ['pdb', 'sequence', 'fasta']
target_pdb_path = '/content/target.pdb' #@param {type:'string'}
target_pdb_chain = 'A' #@param {type:'string'}
target_fasta_path = '/content/target.fasta' #@param {type:'string'}
target_sequence = '' #@param {type:'string'}

binder_csv_path = '' #@param {type:'string'}
binder_name_column = 'Design' #@param {type:'string'}
binder_sequence_column = 'Sequence' #@param {type:'string'}
binder_rank_column = 'Rank' #@param {type:'string'}
binder_fasta_path = '/content/binders.fasta' #@param {type:'string'}
binder_sequence = '' #@param {type:'string'}
binder_name = 'binder_1' #@param {type:'string'}

target_chain_id = 'A' #@param {type:'string'}
binder_chain_id = 'B' #@param {type:'string'}
target_msa_path = '' #@param {type:'string'}
binder_msa_path = '' #@param {type:'string'}

predictions_root = '/content/binder_ipsae_runs/boltz_output' #@param {type:'string'}
use_msa_server = True #@param {type:'boolean'}
use_potentials = False #@param {type:'boolean'}
override_existing = False #@param {type:'boolean'}
devices = 1 #@param {type:'integer'}
model_index = 0 #@param {type:'integer'}
recycling_steps = 0 #@param {type:'integer'}
diffusion_samples = 1 #@param {type:'integer'}
pae_cutoff = 15.0 #@param {type:'number'}
dist_cutoff = 15.0 #@param {type:'number'}

print('Mode:', mode)
print('Work dir:', work_dir)


In [ ]:
#@title Optional file upload
#@markdown Run this if you want to upload `target.pdb`, `target.fasta`, `final_design_stats.csv`, `binders.fasta`, or `.a3m` files directly from your computer.

from google.colab import files

uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))


In [ ]:
ipsae_path = WORKSPACE / 'ipsae.py'
work_dir_path = Path(work_dir)
work_dir_path.mkdir(parents=True, exist_ok=True)

summary_csv_path = work_dir_path / 'ipsae_summary.csv'
manifest_path = work_dir_path / 'binder_manifest.csv'

if mode in {'prepare', 'predict_and_score'}:
    target_seq = load_target_sequence(target_mode, target_sequence, target_fasta_path, target_pdb_path, target_pdb_chain)
    binders = load_binders(binder_sequence, binder_name, binder_fasta_path, binder_csv_path, binder_name_column, binder_sequence_column, binder_rank_column)
    manifest_path = prepare_inputs(
        work_dir=work_dir_path,
        target_sequence=target_seq,
        binders=binders,
        target_name=target_name,
        target_chain_id=target_chain_id,
        binder_chain_id=binder_chain_id,
        target_msa_path=target_msa_path,
        binder_msa_path=binder_msa_path,
    )
    print('Prepared manifest:', manifest_path)
    display(pd.read_csv(manifest_path).head())

if mode == 'predict_and_score':
    recycling_arg = None if recycling_steps <= 0 else recycling_steps
    diffusion_arg = None if diffusion_samples <= 0 else diffusion_samples
    run_boltz(
        input_dir=work_dir_path / 'inputs',
        out_dir=Path(predictions_root),
        use_msa_server=use_msa_server,
        use_potentials=use_potentials,
        override=override_existing,
        devices=devices,
        recycling_steps=recycling_arg,
        diffusion_samples=diffusion_arg,
    )
    summary_df = score_batch(Path(predictions_root), ipsae_path, model_index, pae_cutoff, dist_cutoff)
    summary_df.to_csv(summary_csv_path, index=False)
    print('Saved summary to', summary_csv_path)
    display(summary_df.sort_values(['summary_kind', 'ipSAE'], ascending=[True, False]).head(20))

if mode == 'score_existing_predictions':
    summary_df = score_batch(Path(predictions_root), ipsae_path, model_index, pae_cutoff, dist_cutoff)
    summary_df.to_csv(summary_csv_path, index=False)
    print('Saved summary to', summary_csv_path)
    display(summary_df.sort_values(['summary_kind', 'ipSAE'], ascending=[True, False]).head(20))


In [ ]:
# Inspect the final summary whenever `ipsae_summary.csv` exists.

summary_csv_path = Path(work_dir) / 'ipsae_summary.csv'
if summary_csv_path.exists():
    summary_df = pd.read_csv(summary_csv_path)
    display(summary_df)
    print('Rows:', len(summary_df))
else:
    print('No summary CSV yet at', summary_csv_path)
